# First common forecast origin and Pseudo validation

This notebook converts the fixed maturity rule $(W,k)=(61,10)$ into a shared calendar start for forecast evaluation. It reuses the first-origin design from `01_data_understanding/01_02_active_demand_analysis.ipynb`:

- candidate origins are Mondays;
- all eligibility evidence is restricted to rows with `DATE < candidate`;
- the first origin is the earliest candidate at which at least 80% of known FCM product-store series are mature;
- the seven-calendar-day test horizon must fit inside the extract;
- Pseudo cannot affect the selected origin and is validated only after the FCM stopping rule has selected it.

Pseudo validation uses only history accumulated before the first origin. It checks series, product, store, and retained pre-origin demand coverage.

In [1]:
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)

GROUP_COLS = ["ARTIKEL_ID", "MARKT_ID"]
DEMAND_COL = "ABVERKAUFTE_MENGE_KG"
GROUP_ORDER = ["FCM", "Pseudo"]
CALIBRATION_GROUP = "FCM"
VALIDATION_GROUP = "Pseudo"

MIN_ACTIVE_DAYS = 61
MIN_DEMAND_DAYS = 10
MIN_FCM_ELIGIBILITY_RATE = 0.80
ORIGIN_SPACING_DAYS = 28
FORECAST_HORIZON_CALENDAR_DAYS = 7

# Pseudo validation gates at the selected first origin.
MIN_PSEUDO_SERIES_COVERAGE = 0.95
MIN_PSEUDO_PRODUCT_COVERAGE = 0.95
MIN_PSEUDO_STORE_COVERAGE = 0.95
MIN_PSEUDO_RETAINED_DEMAND = 0.99

PROJECT_ROOT = next(
    (p.resolve() for p in [Path("../.."), Path(".."), Path(".")] if (p / "src").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find the project root containing src/.")

DATA_DIR = PROJECT_ROOT / "data/interim/transactions_dst_daily_min_demand_no_outliers"
parquet_files = sorted(DATA_DIR.glob("*.parquet"))
if not parquet_files:
    raise FileNotFoundError(f"No parquet files found in {DATA_DIR}")
DATA_GLOB = str(DATA_DIR / "*.parquet")

con = duckdb.connect()
con.execute("PRAGMA threads=4")

## Data basis and audit

A present daily row is an observed/open day for its product-store series. Closed dates are not inserted. FCM takes precedence in source assignment, while the audit requires the two source flags not to overlap and each product-store series to retain one source group and reporting category through time.

In [2]:
con.execute(f"""
CREATE OR REPLACE TEMP VIEW daily_rows AS
SELECT
    ARTIKEL_ID,
    MARKT_ID,
    CAST(DATE AS DATE) AS period,
    CAST(COALESCE({DEMAND_COL}, 0) AS DOUBLE) AS demand,
    is_fcm,
    is_pseudo,
    WGR_ID::INTEGER AS category_id,
    CASE
        WHEN is_fcm THEN 'FCM'
        WHEN is_pseudo THEN 'Pseudo'
    END AS sourcing_group
FROM read_parquet('{DATA_GLOB}')
WHERE (is_fcm OR is_pseudo)
  AND WGR_ID IN (890, 900)
""")

row_audit = con.execute("""
SELECT
    COUNT(*) AS rows,
    MIN(period) AS first_date,
    MAX(period) AS last_date,
    COUNT_IF(is_fcm IS NULL) AS null_fcm_flags,
    COUNT_IF(is_pseudo IS NULL) AS null_pseudo_flags,
    COUNT_IF(is_fcm AND is_pseudo) AS overlapping_source_flags
FROM daily_rows
""").fetchdf()
duplicate_series_dates = con.execute("""
SELECT COUNT(*)
FROM (
    SELECT ARTIKEL_ID, MARKT_ID, period
    FROM daily_rows
    GROUP BY ALL
    HAVING COUNT(*) > 1
)
""").fetchone()[0]
changing_series_attributes = con.execute("""
SELECT COUNT(*)
FROM (
    SELECT ARTIKEL_ID, MARKT_ID
    FROM daily_rows
    GROUP BY ARTIKEL_ID, MARKT_ID
    HAVING COUNT(DISTINCT sourcing_group) > 1
        OR COUNT(DISTINCT category_id) > 1
)
""").fetchone()[0]
row_audit["duplicate_series_dates"] = duplicate_series_dates
row_audit["changing_series_attributes"] = changing_series_attributes
display(row_audit)

failure_columns = [
    "null_fcm_flags", "null_pseudo_flags", "overlapping_source_flags",
    "duplicate_series_dates", "changing_series_attributes",
]
if row_audit[failure_columns].sum(axis=1).iloc[0]:
    raise RuntimeError("Source/category audit failed; inspect row_audit.")

population = con.execute("""
SELECT
    sourcing_group,
    COUNT(DISTINCT (ARTIKEL_ID, MARKT_ID)) AS series,
    COUNT(DISTINCT ARTIKEL_ID) AS products,
    COUNT(DISTINCT MARKT_ID) AS stores,
    MIN(period) AS first_date,
    MAX(period) AS last_date
FROM daily_rows
GROUP BY sourcing_group
ORDER BY CASE sourcing_group WHEN 'FCM' THEN 1 ELSE 2 END
""").fetchdf()
display(population)

,rows,first_date,last_date,null_fcm_flags,null_pseudo_flags,overlapping_source_flags,duplicate_series_dates,changing_series_attributes
0,19934887,2023-07-22,2026-07-21,0.0,0.0,0.0,0,0


,sourcing_group,series,products,stores,first_date,last_date
0,FCM,2907,29,190,2025-04-17,2026-07-21
1,Pseudo,22262,342,192,2023-07-22,2026-07-21


## FCM stopping rule

For each Monday candidate, a known series has at least one row before the candidate. It is mature when its pre-origin history contains at least 61 observed/open days and at least 10 positive-demand days. The first origin is selected from FCM alone; Pseudo metrics are carried alongside only for visibility and are not part of the stopping condition.

In [3]:
metrics_cache = {}

def series_metrics_at(cutoff):
    """Summarize each known series using rows strictly before cutoff."""
    cutoff = pd.Timestamp(cutoff).normalize()
    if cutoff not in metrics_cache:
        metrics_cache[cutoff] = con.execute("""
        SELECT
            sourcing_group,
            ARTIKEL_ID,
            MARKT_ID,
            COUNT(*) AS active_days,
            COUNT_IF(demand > 0) AS demand_days,
            SUM(demand) AS cumulative_demand_kg,
            MIN(period) AS first_period,
            MAX(period) AS last_period
        FROM daily_rows
        WHERE period < ?
        GROUP BY sourcing_group, ARTIKEL_ID, MARKT_ID
        """, [cutoff.date()]).fetchdf()
    return metrics_cache[cutoff]


def group_eligibility(metrics):
    rows = []
    for group_name in GROUP_ORDER:
        group = metrics[metrics["sourcing_group"] == group_name]
        eligible = group[
            (group["active_days"] >= MIN_ACTIVE_DAYS)
            & (group["demand_days"] >= MIN_DEMAND_DAYS)
        ]
        total_demand = group["cumulative_demand_kg"].sum()
        retained_demand = eligible["cumulative_demand_kg"].sum()
        rows.append({
            "sourcing_group": group_name,
            "known_series": len(group),
            "eligible_series": len(eligible),
            "eligibility_rate": len(eligible) / len(group) if len(group) else np.nan,
            "product_coverage": (
                eligible["ARTIKEL_ID"].nunique() / group["ARTIKEL_ID"].nunique()
                if group["ARTIKEL_ID"].nunique() else np.nan
            ),
            "store_coverage": (
                eligible["MARKT_ID"].nunique() / group["MARKT_ID"].nunique()
                if group["MARKT_ID"].nunique() else np.nan
            ),
            "retained_pre_origin_demand": (
                retained_demand / total_demand if total_demand > 0 else np.nan
            ),
            "median_active_days": group["active_days"].median(),
            "median_demand_days": group["demand_days"].median(),
        })
    return pd.DataFrame(rows)

In [4]:
extract_dates = con.execute("""
SELECT MIN(period) AS first_date, MAX(period) AS last_date
FROM daily_rows
""").fetchdf().iloc[0]
group_start_dates = con.execute("""
SELECT sourcing_group, MIN(period) AS first_date
FROM daily_rows
GROUP BY sourcing_group
""").fetchdf()

latest_group_start = pd.Timestamp(group_start_dates["first_date"].max())
first_candidate_monday = latest_group_start + pd.Timedelta(
    days=(-latest_group_start.weekday()) % 7
)
latest_test_start = pd.Timestamp(extract_dates["last_date"]) - pd.Timedelta(
    days=FORECAST_HORIZON_CALENDAR_DAYS - 1
)
last_complete_monday = latest_test_start - pd.Timedelta(
    days=latest_test_start.weekday()
)
candidate_mondays = pd.date_range(
    first_candidate_monday, last_complete_monday, freq="W-MON"
)

search_rows = []
first_origin = None
first_origin_metrics = None
first_origin_diagnostics = None

for candidate in candidate_mondays:
    current_metrics = series_metrics_at(candidate)
    diagnostics = group_eligibility(current_metrics).set_index("sourcing_group")
    if not set(GROUP_ORDER).issubset(diagnostics.index):
        continue
    search_rows.append({
        "candidate_monday": candidate,
        "known_fcm_series": int(diagnostics.loc["FCM", "known_series"]),
        "eligible_fcm_series": int(diagnostics.loc["FCM", "eligible_series"]),
        "fcm_eligibility_rate": diagnostics.loc["FCM", "eligibility_rate"],
        "pseudo_eligibility_rate_diagnostic": diagnostics.loc[
            "Pseudo", "eligibility_rate"
        ],
    })
    if diagnostics.loc[CALIBRATION_GROUP, "eligibility_rate"] >= MIN_FCM_ELIGIBILITY_RATE:
        first_origin = candidate
        first_origin_metrics = current_metrics
        first_origin_diagnostics = diagnostics.reset_index()
        break

if first_origin is None:
    raise RuntimeError(
        "No complete Monday reaches the FCM maturity-coverage requirement."
    )

origin_search = pd.DataFrame(search_rows)
forecast_origins = pd.date_range(
    first_origin, last_complete_monday, freq=f"{ORIGIN_SPACING_DAYS}D"
)
assert len(forecast_origins) > 0
assert (forecast_origins.weekday == 0).all()
origin_schedule = pd.DataFrame({"origin": forecast_origins})
origin_schedule["test_end_inclusive"] = (
    origin_schedule["origin"]
    + pd.Timedelta(days=FORECAST_HORIZON_CALENDAR_DAYS - 1)
)
display(group_start_dates.sort_values("sourcing_group"))
display(origin_search.style.format({
    "known_fcm_series": "{:,.0f}", "eligible_fcm_series": "{:,.0f}",
    "fcm_eligibility_rate": "{:.1%}",
    "pseudo_eligibility_rate_diagnostic": "{:.1%}",
}))
print(
    f"Selected first origin: {first_origin.date()} ({first_origin.day_name()}); "
    f"FCM eligibility = "
    f"{first_origin_diagnostics.set_index('sourcing_group').loc['FCM', 'eligibility_rate']:.1%}."
)
print(
    f"Number of {ORIGIN_SPACING_DAYS}-day-spaced origins: "
    f"{len(forecast_origins)}."
)
display(origin_schedule)

,sourcing_group,first_date
1,FCM,2025-04-17
0,Pseudo,2023-07-22


,candidate_monday,known_fcm_series,eligible_fcm_series,fcm_eligibility_rate,pseudo_eligibility_rate_diagnostic
0,2025-04-21 00:00:00,58,0,0.0%,97.2%
1,2025-04-28 00:00:00,127,0,0.0%,97.3%
2,2025-05-05 00:00:00,143,0,0.0%,97.4%
3,2025-05-12 00:00:00,327,0,0.0%,97.5%
4,2025-05-19 00:00:00,424,0,0.0%,97.6%
5,2025-05-26 00:00:00,478,0,0.0%,97.5%
6,2025-06-02 00:00:00,662,0,0.0%,97.2%
7,2025-06-09 00:00:00,955,0,0.0%,97.3%
8,2025-06-16 00:00:00,"1,096",0,0.0%,97.4%
9,2025-06-23 00:00:00,"1,153",0,0.0%,97.8%


Selected first origin: 2025-12-01 (Monday); FCM eligibility = 85.0%.
Number of 28-day-spaced origins: 9.


,origin,test_end_inclusive
0,2025-12-01,2025-12-07
1,2025-12-29,2026-01-04
2,2026-01-26,2026-02-01
3,2026-02-23,2026-03-01
4,2026-03-23,2026-03-29
5,2026-04-20,2026-04-26
6,2026-05-18,2026-05-24
7,2026-06-15,2026-06-21
8,2026-07-13,2026-07-19


## Pseudo validation at the selected origin

The selected date is already fixed by FCM. Pseudo is now checked using the same strict `DATE < first_origin` boundary and the same `(61, 10)` maturity rule. The validation requires near-total coverage of Pseudo series, products, stores, and cumulative pre-origin demand. It does not inspect future test outcomes and cannot move the origin.

In [5]:
display(first_origin_diagnostics.style.format({
    "known_series": "{:,.0f}", "eligible_series": "{:,.0f}",
    "eligibility_rate": "{:.1%}", "product_coverage": "{:.1%}",
    "store_coverage": "{:.1%}", "retained_pre_origin_demand": "{:.1%}",
    "median_active_days": "{:,.1f}", "median_demand_days": "{:,.1f}",
}))

pseudo = first_origin_diagnostics.set_index("sourcing_group").loc[VALIDATION_GROUP]
pseudo_checks = {
    "series coverage": pseudo["eligibility_rate"] >= MIN_PSEUDO_SERIES_COVERAGE,
    "product coverage": pseudo["product_coverage"] >= MIN_PSEUDO_PRODUCT_COVERAGE,
    "store coverage": pseudo["store_coverage"] >= MIN_PSEUDO_STORE_COVERAGE,
    "retained pre-origin demand": (
        pseudo["retained_pre_origin_demand"] >= MIN_PSEUDO_RETAINED_DEMAND
    ),
}
display(pd.Series(pseudo_checks, name="passed").to_frame())
if not all(pseudo_checks.values()):
    failed = [name for name, passed in pseudo_checks.items() if not passed]
    raise RuntimeError(f"Pseudo first-origin validation failed: {', '.join(failed)}")
print("Pseudo first-origin validation: PASS")

,sourcing_group,known_series,eligible_series,eligibility_rate,product_coverage,store_coverage,retained_pre_origin_demand,median_active_days,median_demand_days
0,FCM,"2,119","1,801",85.0%,83.3%,98.9%,95.6%,144.0,43.0
1,Pseudo,"22,140","21,981",99.3%,99.4%,98.9%,100.0%,714.0,141.0


,passed
series coverage,True
product coverage,True
store coverage,True
retained pre-origin demand,True


Pseudo first-origin validation: PASS


## Information boundary and result

The audit below verifies that the latest row used for origin selection and Pseudo validation is strictly earlier than the selected Monday. The output date is the shared first origin for downstream forecast evaluation.

In [6]:
latest_information_date = con.execute(
    "SELECT MAX(period) FROM daily_rows WHERE period < ?", [first_origin.date()]
).fetchone()[0]
assert pd.Timestamp(latest_information_date) < first_origin
assert first_origin.weekday() == 0

result = pd.DataFrame([{
    "maturity_W": MIN_ACTIVE_DAYS,
    "maturity_k": MIN_DEMAND_DAYS,
    "first_origin": first_origin,
    "number_of_origins": len(forecast_origins),
    "origin_spacing_days": ORIGIN_SPACING_DAYS,
    "weekday": first_origin.day_name(),
    "latest_information_date": pd.Timestamp(latest_information_date),
    "test_end_inclusive": first_origin + pd.Timedelta(
        days=FORECAST_HORIZON_CALENDAR_DAYS - 1
    ),
}])
display(result)
print(f"Final first common forecast origin: {first_origin:%A, %d %B %Y}")

,maturity_W,maturity_k,first_origin,number_of_origins,origin_spacing_days,weekday,latest_information_date,test_end_inclusive
0,61,10,2025-12-01,9,28,Monday,2025-11-30,2025-12-07


Final first common forecast origin: Monday, 01 December 2025
